# Phase 7 — Regression Benchmarks and Linear Baseline

## TL;DR

- The global-median benchmark has a validation MAE of about 57.8k hg/ha.
- A stronger crop-median benchmark reduces validation MAE to about 36.6k hg/ha.
- Linear regression reduces validation MAE further to about 30.6k hg/ha and explains about 71.2% of validation yield variation.
- Linear regression beats both benchmarks, but it produces negative yield predictions. It is therefore a useful baseline, not yet a production model.
- The 2011–2013 test set remains untouched.


## Context & Methods

Three models are fitted on 1990–2007 training data and compared on 2008–2010 validation data:

1. a global-median benchmark;
2. a crop-specific median benchmark with a global fallback; and
3. ordinary least-squares linear regression using the leakage-safe preprocessing pipeline.

### Key assumptions

- Lower MAE and RMSE are better.
- Higher R² is better, but R² can be negative for a poor model.
- Model selection uses validation only; test is reserved for final evaluation.
- Predictive associations must not be described as causal effects.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crop_yield.evaluation import (
    prediction_diagnostics,
    regression_metrics,
)
from crop_yield.models import (
    CropMedianRegressor,
    build_global_median_baseline,
    build_linear_regression_pipeline,
)
from crop_yield.preprocessing import split_features_target
from crop_yield.splitting import temporal_train_validation_test_split

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Data

### 1. Load and split before modelling


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)
temporal_split = temporal_train_validation_test_split(crop_yield)

X_train, y_train = split_features_target(temporal_split.train)
X_validation, y_validation = split_features_target(
    temporal_split.validation
)

print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Reserved test rows: {len(temporal_split.test):,}")


## Results

### 2. Fit benchmarks and linear regression

Linear regression chooses coefficients that minimize the sum of squared training errors. Its preprocessing and model are kept in one pipeline.


In [ ]:
models = {
    "Global median": build_global_median_baseline(),
    "Crop median": CropMedianRegressor(),
    "Linear regression": build_linear_regression_pipeline(),
}

validation_predictions = {}
validation_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)
    validation_predictions[model_name] = predictions
    validation_results.append(
        {
            "model": model_name,
            **regression_metrics(y_validation, predictions),
        }
    )

results = (
    pd.DataFrame(validation_results)
    .set_index("model")
    .sort_values("mae")
)
results


### 3. Compare validation errors visually


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

results.sort_values("mae")["mae"].plot.barh(
    ax=axes[0],
    color="#3B6EA8",
)
axes[0].set_title("Validation MAE")
axes[0].set_xlabel("Mean absolute error (hg/ha)")
axes[0].set_ylabel("")

results.sort_values("rmse")["rmse"].plot.barh(
    ax=axes[1],
    color="#8FA8C7",
)
axes[1].set_title("Validation RMSE")
axes[1].set_xlabel("Root mean squared error (hg/ha)")
axes[1].set_ylabel("")

fig.suptitle("Benchmark and Linear-Regression Comparison", fontsize=16)
fig.tight_layout()
plt.show()


### 4. Quantify improvement over the stronger benchmark

The crop-median benchmark is the more meaningful reference because crop type strongly separates yield.


In [ ]:
crop_median_mae = results.loc["Crop median", "mae"]
linear_mae = results.loc["Linear regression", "mae"]
mae_improvement = (crop_median_mae - linear_mae) / crop_median_mae

print(
    "Linear regression reduces MAE relative to the crop-median "
    f"benchmark by {mae_improvement:.1%}."
)


### 5. Check physical plausibility

Yield cannot be negative. Negative predictions reveal a limitation of unconstrained ordinary linear regression even when its average metrics are stronger.


In [ ]:
linear_diagnostics = prediction_diagnostics(
    validation_predictions["Linear regression"]
)
pd.Series(linear_diagnostics)


### 6. Confirm that test remains untouched


In [ ]:
test_evaluated = False
print(
    "Test evaluation has not been performed. "
    f"Reserved observations: {len(temporal_split.test):,}."
)


## Checks


In [ ]:
assert results.loc["Linear regression", "mae"] < results.loc["Crop median", "mae"]
assert results.loc["Crop median", "mae"] < results.loc["Global median", "mae"]
assert np.isclose(results.loc["Global median", "mae"], 57_844.32, atol=1.0)
assert np.isclose(results.loc["Crop median", "mae"], 36_630.15, atol=1.0)
assert np.isclose(results.loc["Linear regression", "mae"], 30_582.92, atol=1.0)
assert np.isclose(results.loc["Linear regression", "r2"], 0.7121, atol=0.001)
assert linear_diagnostics["negative_prediction_count"] > 0
assert test_evaluated is False
print("All Phase 7 baseline-model checks passed.")


## Takeaways

1. Linear regression is meaningfully better than both validation benchmarks.
2. Its MAE is about 30.6k hg/ha, which means predictions miss observed yield by that amount on average.
3. RMSE remains higher than MAE, showing that some errors are substantially larger than the typical absolute error.
4. Negative predictions are physically impossible and prevent the current linear baseline from being selected for production.
5. The next experiment should compare approaches that better represent nonlinearity and non-negative yield, while continuing to select models on validation only.
